# report08 — 드론 RCS·마이크로도플러 결과

> ### ❓ 이 리포트가 답하는 질문
> **드론 5종은 실제로 얼마나 밝고, 프로펠러는 어떤 지문을 남기나?**

### ⚡ 결론부터 (TL;DR)

1. **밝기를 정하는 건 기체 크기지 주파수가 아니다.** 5종을 세 통신대역(1.8·3.5·5.2 GHz)에서 재보면, 가장 큰 DJI S1000+(1045 mm)와 가장 작은 DJI Mini 5 Pro(275 mm)의 방위평균 밝기 차이가 **10.2 dB** 나 된다. 반면 같은 드론을 대역만 바꿔 재면 **평균 1.6 dB** 밖에 안 움직인다 — 드론은 이미 파장보다 훨씬 커서(광학영역) 밝기가 대략 **투영 넓이**를 따라간다.
2. **밝은 건 껍데기가 아니라 속 금속이다.** DJI Mavic 4 Pro를 부품별로 벗겨 재보면, 플라스틱 셸을 **지워도** 밝기가 오히려 **+2.0 dB** 올라간다(셸이 약한 가림막). 모터·배터리·PCB 같은 **금속 코어만 남겨도 +2.0 dB** — 통드론과 거의 같다. 반대로 **금속을 다 빼면 -5.5 dB** 어두워진다. 껍데기는 스크린, 표적은 그 안의 금속이다.
3. **방위 패턴에서 인용할 건 '봉우리'뿐이다.** 코·꼬리·측면 같은 로브(봉우리)는 안정적이라 숫자로 옮겨도 되지만, 로브 사이의 **골(널)은 격자·평활에 10 dB 넘게 흔들려** 인용하면 안 된다. 그래서 우리는 **방위평균**과 **로브**만 링크버짓에 쓴다.
4. **프로펠러가 돌면 '깜빡이는' 지문이 생긴다 — 마이크로도플러.** 블레이드가 정면을 보일 때마다 반사가 번쩍인다. 그 번쩍임 주기(flash rate)와 날개끝 도플러 폭(f_tip)은 **호버 회전수**만 알면 계산된다. 회전수는 추력=무게 균형(T=C_T ρ n² D⁴, C_T≈0.11)에서 유도한다 — 183~120 Hz 번쩍임, ±1.0~1.6 kHz 날개끝 도플러.
5. **블레이드 지문을 제대로 보려면 '가림'이 필수다.** 몸통 뒤로 돌아간 블레이드는 안 보여야 하는데(가림), 가림 없는 순수 PO 는 숨은 날개까지 세어 **정지 몸통 신호(pedestal)를 9~23 dB 부풀린다** → 깜빡임이 그 아래 묻힌다. SBR 은 광선이 첫 충돌에서 멈춰 가림이 공짜라, 블레이드 깜빡임을 몸통 신호 위로 되살린다.

### 🗺️ 어디부터 읽나

| 절 | 무엇을 |  |
|---|---|---|
| §1 | **5종 얼마나 밝나** — 기체 크기가 밝기를 정한다 | RCS 결과 헤드라인. 링크버짓에 쓸 숫자 |
| §2 | **밝기는 속 금속이 지배** — 껍데기는 스크린 | '어디가 되비추나' + 정직한 한계(반투명 셸) |
| §3 | **방위 패턴** — 로브는 인용, 널은 인용 금지 | 어떤 숫자를 믿어도 되나 |
| §4 | **프로펠러 지문 = 마이크로도플러** — flash·f_tip·가림 | 드론을 새·잡음과 가르는 축 |
| 바쁘면 | §1 그림 (report2_rcs_bars.png) + §4 (report1_microdoppler.png) | '얼마나 밝나' 와 '어떤 지문' 이 각각 한 장에 |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 드론 물리 스펙 (대각·무게·프로펠러·로터 수) | DJI 공식 제품 스펙 (Mini 5 Pro · Mavic 4 Pro · Matrice 4E · S1000+ · Phantom 4) | 📄 제조사 스펙 |
| 5종 RCS (3밴드) · 재질 분해 | **`outputs/report2_waveform_rcs.json`** 의 `rcs` / `materials`. `src/viz_report2.py` 가 `src/rcs_sbr.py`(SBR)를 돌려 남긴다 | 🟡 측정 (SBR = Mitsuba 광선 + PO) |
| 호버 rpm 유도 · 블레이드 마이크로도플러 | **`outputs/report1.json`** 의 `articulation`(추력 균형) / `microdoppler`. `src/viz_report3.py` + `src/microdoppler.py` 가 남긴다 | 🟡 측정 (자세별 SBR 산란장) |
| 재질별 반사계수 | **`src/materials.py`** — Sionna RT 와 SBR 이 함께 읽는 단일 진리원 (ITU-R P.2040 기반 + custom) | 📐 물성표 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `microdoppler` | 마이크로도플러 (`src/microdoppler.py`) — 회전 블레이드의 슬로타임 복소장 → STFT | 🟡 **우리가 짰다** — 자세별 산란장은 SBR(Mitsuba 광선)로 계산 (GPU) |
| `sionna-render` | Sionna RT `Scene.render_to_file()` — 씬·**추적된 광선**·라디오맵을 사진처럼 렌더 | 🟢 **Sionna 내부** (Mitsuba 3 경로추적 렌더러, GPU) |
| `po` | 순수 물리광학 (`src/rcs_po.py`) — 점구름 PO. **가림 없음** | 🔴 **별도** (numpy, CPU). **비교·검증용으로만** 남겨둠 — 기본 엔진은 SBR |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `drjit` | 1.3.1 | Mitsuba 의 JIT 컴파일러 — GPU 커널 생성 |
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: 5종 × 3밴드 RCS 는 GPU 한 장(#2)에서 수십 분(광선격자 λ/16). 마이크로도플러는 자세 144개 × SBR 재계산이라 드론당 수~십수 분.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2

# 5종 RCS(3밴드) + 재질 분해  -> report2_waveform_rcs.json
~/.venvs/py312/bin/python src/viz_report2.py

# 호버 rpm 유도 + 블레이드 마이크로도플러  -> report1.json
~/.venvs/py312/bin/python src/viz_report3.py

# JSON -> report08.ipynb (이 파일)
~/.venvs/py312/bin/python src/make_notebook08.py
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report2_waveform_rcs.json` | **이 노트북의 RCS 숫자.** rcs(5종×3밴드) / materials(재질 분해) |
| `outputs/report1.json` | **이 노트북의 마이크로도플러 숫자.** articulation(호버 rpm) / microdoppler(지문) |
| `outputs/figures/report2_rcs_bars.png` | §1 5종 밝기 · 크기 추세 |
| `outputs/figures/report2_materials.png` | §2 재질 분해 (껍데기 vs 금속) |
| `outputs/figures/report2_rcs_polar.png` | §3 방위 패턴 (로브 vs 널) |
| `outputs/figures/report1_hover_rpm.png` | §4 호버 rpm 유도 |
| `outputs/figures/report1_microdoppler.png` | §4 블레이드 지문 + 가림 대가 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **절대 RCS 값을 보장하지 않는다.** SBR 은 해석해(구·평판)로만 검증됐고 드론 실측 앵커링이 없다(방법 검증은 report07). 이 리포트가 지지하는 것은 **상대 순서**(큰 기체가 밝다)와 **대역 추세**(밴드는 몇 dB만 움직인다)이지 특정 드론의 절대 dBsm 이 아니다.
- **플라스틱 셸의 밝기는 불확실 구간이다.** 1~3 mm 셸은 1.8~5.2 GHz 에서 **반투명**인데 first-hit SBR 은 셸을 뚫지 못한다. 그래서 진실은 '통드론'과 '셸 제거' 두 막대 사이에 있고, 그 간격 **2.0 dB** 는 측정오차가 아니라 **모델링 불확실도**로 읽어야 한다.
- **방위 패턴의 '널(골)'은 인용 금지.** 로브 사이 골은 격자밀도·대역평균·평활에 10 dB 넘게 흔들린다. **로브(봉우리)와 방위평균만** 믿는다.
- **호버 rpm 은 가정값이다.** 추력=무게 균형(C_T≈0.11)에서 유도한 물리 추정치이지 텔레메트리 실측이 아니다. flash·f_tip 은 이 rpm 에 선형으로 비례하므로, 실제 비행 rpm 이 다르면 지문 주파수도 그만큼 이동한다.
- **마이크로도플러는 슬로타임 모델이다.** 자세별 산란장은 SBR(Mitsuba 광선)로 재계산하지만, 블레이드 유연·와류 등 공기역학은 넣지 않았다. 지문의 **구조**(깜빡임·f_tip 경계)는 믿을 만 하나 절대 세기는 아니다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| **앞** — [report07](report07.ipynb) | 이 숫자를 낸 **방법(SBR)** — 왜 옳은가·가림이 무엇인가 |
| **다음** — [report09](report09.ipynb) | 이제 탐지로. 먼저 챔버 **바닥이 놓는 함정**(표적 경유 유령) |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **RCS (σ)** | 레이더 되비침 밝기 [m²]. '이 표적이 얼마나 밝게 되쏘나'. dBsm = 10·log₁₀(σ/1 m²) |
| **dBsm** | 1 m² 대비 dB. −20 dBsm = 0.01 m² = 되비침이 사방 10 cm 판 만큼 |
| **광학영역** | 표적이 파장보다 훨씬 클 때. 밝기가 대략 **투영 넓이**를 따라가고 주파수엔 둔감 |
| **SBR** | 광선을 쏴 보이는 면을 찾고(가림 처리) 그 위에서 밝기를 위상 맞춰 적분. 우리 RCS 엔진 |
| **가림(occlusion)** | 앞 부품에 막혀 안 보이는 면. 이걸 안 빼면 밝기·정지신호를 과대평가한다 |
| **로브 / 널** | 방위 패턴의 봉우리(로브)와 골(널). 로브는 안정, 널은 불안정 → 널은 인용 금지 |
| **마이크로도플러** | 표적의 **부분 운동**(프로펠러 회전)이 만드는 도플러 미세구조. 드론의 지문 |
| **flash rate** | 블레이드가 정면을 보여 번쩍이는 초당 횟수 = 날개수 × 회전수/60 |
| **f_tip** | 날개 끝 속도가 만드는 최대 도플러 폭. f_tip = 2·v_tip/λ·cos(el) |
| **pedestal(정지 몸통 신호)** | 회전 안 하는 몸통이 만드는 0 Hz 근처 강한 성분. 블레이드 깜빡임은 이 위로 솟아야 보인다 |
| **C_T (추력계수)** | 프로펠러 추력을 회전수로 잇는 무차원 계수. T = C_T ρ n² D⁴, 소형 로터 ≈0.11 |

</details>

---


## 🔰 5분이면 이해하는 이 리포트

**한 줄:** 드론 5종이 레이더 눈에 얼마나 **밝게** 보이는지, 그리고 프로펠러가 어떤 **깜빡이는 지문**을 남기는지를 잽니다.

**밝기 이야기 — 비유로.** 어두운 방에서 물건에 손전등을 비추고 얼마나 되쏘는지 본다고 합시다.

- **큰 금속판**이 **작은 플라스틱 조각**보다 훨씬 밝게 반짝입니다. 드론도 똑같습니다 — **기체가 클수록 밝습니다.** 가장 큰 DJI S1000+는 가장 작은 DJI Mini 5 Pro보다 **10 dB**(십 배 남짓) 밝습니다.
- 놀라운 점: **손전등 색(=전파 주파수)을 바꿔도** 밝기는 별로 안 변합니다(몇 dB). 드론이 이미 파장보다 훨씬 크기 때문입니다.
- 더 놀라운 점: **밝은 건 플라스틱 껍데기가 아니라 그 안의 금속**(모터·배터리·기판)입니다. 플라스틱 껍데기는 빛을 살짝 가리는 **반투명 커튼**일 뿐, 진짜로 반짝이는 건 속 금속입니다.

**지문 이야기 — 비유로.** 이번엔 **돌아가는 선풍기 날개**에 손전등을 비춰 봅시다.

- 날개가 정면을 향하는 **순간마다 번쩍**입니다. 이 규칙적인 번쩍임이 **초당 몇 번**이냐가 드론마다 다릅니다(프로펠러가 크고 느리면 드물게, 작고 빠르면 자주). 이게 프로펠러의 지문입니다.
- 정지한 물체(벽·새)는 이런 깜빡임이 없습니다. 그래서 이 지문은 **드론을 잡음·클러터와 가르는 축**이 됩니다.
- 단, 몸통 뒤로 돌아간 날개는 **안 보여야** 합니다(가림). 이 가림을 빼먹으면 몸통의 밋밋한 신호가 부풀어 올라 정작 중요한 **깜빡임이 그 아래 묻혀** 버립니다. 우리 방법(SBR)은 광선이 맨 앞에서 멈추므로 가림이 저절로 처리돼, 깜빡임을 몸통 위로 되살립니다.

> 한 마디로: **드론은 크기가 밝기를 정하고(속 금속이 주역), 프로펠러는 회전이 만드는 규칙적 깜빡임으로 자기를 드러냅니다.**

> 📎 **앞 리포트([report07](report07.ipynb))에서** 이 밝기를 어떻게 재는지(SBR — 광선 위에 적분 얹기, 가림 공짜)를 설명했습니다. 이 리포트는 그 방법으로 **실제로 잰 결과**입니다.

## 1. 5종 얼마나 밝나 — 밝기를 정하는 건 크기지 주파수가 아니다

![rcs bars](outputs/figures/report2_rcs_bars.png)

각 드론을 **360° 다 돌려가며**(방위) 세 통신대역에서 재고, 그 **방위평균**을 밝기 대표값으로 씁니다. (봉우리 값이 아니라 평균입니다 — 링크버짓에 넣을 정직한 숫자는 이쪽입니다.)

**밴드평균 방위평균 RCS [dBsm]** (el = 15°, 격자 λ/16):

| 드론 | 대각 [mm] | 무게 [g] | LTE 1.8 GHz | 5G NR 3.5 GHz | WiFi 5.2 GHz |
|---|---|---|---|---|---|
| DJI Mini 5 Pro | 275 | 250 | -25.9 | -24.5 | -24.2 |
| DJI Mavic 4 Pro | 441 | 1063 | -19.7 | -19.7 | -18.6 |
| DJI Matrice 4E | 438.8 | 1219 | -24.1 | -24.1 | -24.1 |
| DJI S1000+ | 1045 | 9500 | -16.6 | -13.8 | -13.7 |
| DJI Phantom 4 | 350 | 1380 | -24.4 | -23.3 | -22.4 |

**두 가지가 한눈에 보입니다.**

1. **크기가 밝기를 정합니다.** 가장 큰 DJI S1000+(대각 1045 mm, 9.5 kg 8로터)가 가장 밝고, 가장 작은 DJI Mini 5 Pro(275 mm, 250 g)가 가장 어둡습니다. 둘 사이가 **10.2 dB** — 퍼센트 수준이 아니라 **십 배 남짓** 차이입니다. 오른쪽 산점도가 대각↔밝기 추세를 그대로 보여줍니다.

2. **대역(주파수)은 별로 안 움직입니다.** 같은 드론을 1.8 → 5.2 GHz 로 옮겨도 밝기는 평균 **1.6 dB** 밖에 안 변합니다. 드론이 이미 파장(λ = 6~17 cm)보다 훨씬 커서 **광학영역**에 있기 때문입니다 — 이 영역에선 밝기가 대략 **투영 넓이**를 따라가고 파장엔 둔감합니다.

> 🔎 **왜 크기 순서가 무게 순서와 살짝 다른가?** RCS 는 무게가 아니라 **되비추는 금속 표면**이 정합니다. DJI Phantom 4는 무겁지만(1.38 kg) 몸체가 매끈해 측면 로브가 좁고, DJI Mini 5 Pro는 250 g 급이라 되쏠 금속 자체가 작습니다. 순서는 **투영된 금속 넓이** 쪽을 따릅니다.

> 📌 **이 표에서 링크버짓으로 가져갈 숫자는 방위평균입니다.** 봉우리(peak)는 순간적으로 더 밝지만 방위가 조금만 틀어져도 사라집니다 — 탐지 성능을 보수적으로 보려면 평균을 씁니다.

## 2. 밝기는 속 금속이 지배한다 — 껍데기는 스크린이다

![materials](outputs/figures/report2_materials.png)

밝기가 **어디서** 나오는지 보려면 드론을 부품별로 벗겨가며 재보면 됩니다. DJI Mavic 4 Pro 한 대를 3.5 GHz 에서, 부품을 하나씩 지우며 방위평균 밝기를 다시 쟀습니다(밝은 쪽이 위):

| 무엇을 남겼나 | 방위평균 RCS | 통드론 대비 |
|---|---|---|
| **통드론** (플라스틱 셸 포함) | -19.77 dBsm | 기준 |
| 셸 **제거** (전파가 플라스틱을 통과) | -17.80 dBsm | **+1.97 dB** |
| 프로펠러만 제거 | -19.83 dBsm | -0.06 dB |
| **금속 코어만** (모터+배터리+PCB+카메라) | -17.79 dBsm | **+1.98 dB** |
| 유전체만 (금속 하나도 없이) | -25.23 dBsm | -5.47 dB |

**세 줄로 요약됩니다.**

- **플라스틱 껍데기를 지웠더니 오히려 +1.97 dB 밝아졌습니다.** 셸은 밝기에 거의 기여하지 않으면서, 뒤에 있는 금속으로 갈 광선을 약하게 가로막던 **가림막**이었기 때문입니다. (지운다는 건 페인트를 칠하는 게 아니라 그 면을 메쉬에서 **삭제**해 전파가 통과하게 하는 것입니다.)
- **금속 코어만 남겨도 +1.98 dB** — 통드론과 사실상 같습니다. 반대로 **금속을 전부 빼면 -5.47 dB** 어두워집니다. **밝기를 만드는 건 속 금속이고, 플라스틱은 조연**입니다.
- 프로펠러(플라스틱)는 정지 상태에서 **-0.06 dB** — 밝기엔 거의 무의미합니다. (단, **돌면** 이야기가 완전히 달라집니다 → §4.)

> ⚠️ **정직한 한계 하나.** 1~3 mm 플라스틱 셸은 1.8~5.2 GHz 에서 실제로는 **반투명**입니다 — 전파가 얼마쯤 통과합니다. 그런데 우리 SBR 은 광선이 **첫 충돌에서 멈추므로** 셸을 뚫지 못합니다. 그래서 진실은 '통드론(불투명 셸)'과 '셸 제거(투명 셸)' **두 막대 사이 어딘가**에 있습니다. 그 간격 **2.0 dB** 는 측정오차가 아니라 **모델링 불확실도**로 읽으십시오.

> 🔗 두 엔진(전파용 Sionna RT · RCS용 SBR)이 **같은 재질표**(`src/materials.py`)를 읽습니다. 오른쪽 표의 반사계수가 그것 — 조용히 어긋날 수 없습니다.

## 3. 방위 패턴 — '봉우리'는 인용, '골'은 인용 금지

![rcs polar](outputs/figures/report2_rcs_polar.png)

드론을 한 바퀴 돌리면 밝기는 방위에 따라 **꽃잎 모양**으로 오르내립니다. 코(0°)·꼬리(180°)·측면(90°/270°)에서 넓은 금속면이 정면으로 보일 때 **봉우리(로브)** 가 서고, 그 사이에서 **골(널)** 로 떨어집니다.

**여기서 반드시 지켜야 할 규칙:**

- **봉우리(로브)는 인용해도 됩니다.** 위치와 높이가 격자밀도·대역평균·평활을 바꿔도 안정적입니다. 링크버짓의 '최선의 경우'로 쓸 수 있습니다.
- **골(널)은 절대 인용하지 마십시오.** 골의 깊이는 여러 반사가 서로 상쇄돼 생기는 것이라, 격자를 조금만 바꿔도 **10 dB 넘게** 출렁입니다. '이 각도에서 −40 dBsm 으로 안 보인다' 같은 주장은 하면 안 됩니다.

> 그래서 이 리포트가 밖으로 내보내는 숫자는 **§1 의 방위평균**과 **로브 높이**뿐입니다. 특정 방위의 널 깊이는 내부 그림에서만 봅니다.

> 📐 각 곡선은 361개 방위 × 대역 내 5개 주파수 평균 × 3° 평활입니다(el = 15°). 큰 기체(S1000+)일수록 로브가 잘게 갈라지는 건 전기적 크기(size/λ)가 커서 로브가 촘촘해지기 때문입니다.

![.](outputs/renders/anim/rcs_azimuth_matrice4e.gif)

<sub>Matrice 4E RCS 방위각 폴라 — 각도마다 수 dB~수십 dB 출렁인다(SBR 결과).</sub>

## 4. 프로펠러 지문 — 마이크로도플러

지금까지는 드론이 **가만히** 있을 때의 밝기였습니다. 하지만 드론의 프로펠러는 초당 수십 바퀴를 돕니다. 돌아가는 블레이드는 **정면을 보일 때마다 반사가 번쩍**이고, 날개 끝은 시속 200 km 급으로 움직여 큰 도플러(주파수 변화)를 만듭니다. 이 미세구조가 드론을 새·잡음과 가르는 **지문**입니다.

### 4.1 지문의 두 눈금은 호버 회전수에서 나온다

![hover rpm](outputs/figures/report1_hover_rpm.png)

지문에는 두 개의 눈금이 있습니다.

- **flash rate(번쩍임 주기)** = 날개수 × 회전수/60. 2엽 프로펠러는 한 바퀴에 정면을 **두 번** 보이므로 flash = 회전수/30. → 프로펠러가 **크고 느린** 기체는 드물게, **작고 빠른** 기체는 자주 번쩍입니다.
- **f_tip(날개끝 도플러 폭)** = 2·v_tip/λ·cos(el). 날개 끝 속도 v_tip = ω·R 가 만드는 **최대** 도플러입니다. 모델 안에서 이보다 빨리 움직이는 산란체는 없으므로, 진짜 마이크로도플러는 **±f_tip 안에** 갇힙니다.

둘 다 **호버 회전수**만 알면 정해집니다. 회전수는 텔레메트리가 없으니 **물리로 유도**합니다 — 호버란 4(또는 8)개 로터의 추력이 정확히 무게를 받치는 상태이고, 추력은 T = C_T ρ n² D⁴ (C_T ≈ 0.10~0.12) 로 회전수 n 과 이어집니다. 무게와 프로펠러 지름 D 를 넣어 n 을 풀면 **아래 표의 호버 rpm**이 나옵니다(가정값이지만 물리 범위 안입니다).

**5종의 프로펠러 지문** (3.5 GHz, el = 15°):

| 드론 | 로터 수 | 호버 rpm | flash [Hz] | f_tip [kHz] | 가림 이득 [dB] |
|---|---|---|---|---|---|
| DJI Mini 5 Pro | 4 | 5500 | 183 | ±0.99 | 14 |
| DJI Mavic 4 Pro | 4 | 3600 | 120 | ±1.14 | 22 |
| DJI Matrice 4E | 4 | 3800 | 127 | ±1.23 | 14 |
| DJI S1000+ | 8 | 3600 | 120 | ±1.62 | 9 |
| DJI Phantom 4 | 4 | 5500 | 183 | ±1.56 | 23 |

→ **flash rate 만으로도 기체가 갈립니다.** 큰 프로펠러(S1000+ 15인치)는 느리게 돌아 120 Hz, 작은 프로펠러(Mini 5 Pro 6인치)는 빠르게 돌아 183 Hz 로 번쩍입니다. 지문 주파수가 곧 기체 식별자입니다.

### 4.2 지문을 보려면 '가림'이 필수다

![microdoppler](outputs/figures/report1_microdoppler.png)

위 그림의 각 판은 시간(가로) × 도플러(세로)로 그린 **슬로타임 반사장**입니다. 프레임마다 블레이드 자세를 다시 놓고 SBR 로 산란장을 새로 계산합니다 — 세로 줄무늬가 바로 블레이드 번쩍임, 파란 점선이 ±f_tip 경계입니다.

여기서 **가림이 왜 필수인지**가 오른쪽 아래 막대에 있습니다. 블레이드가 몸통 뒤로 돌아가면 **안 보여야** 하는데, 가림을 안 하는 순수 PO 는 **숨은 날개와 셸 속 금속까지 다 세어** 0 Hz 근처의 **정지 몸통 신호(pedestal)를 부풀립니다.** 그 부풀림이 드론마다 **9~23 dB** — 그만큼 블레이드 깜빡임이 몸통 신호 아래 묻힙니다.

SBR 은 광선이 **첫 충돌에서 멈춰** 가림이 공짜라, 부풀린 pedestal 을 걷어내고 **깜빡임을 몸통 위로 되살립니다.** 그래서 §2 에서 '정지 상태 프로펠러는 밝기에 무의미'했지만, **돌면** 프로펠러가 지문의 주역이 됩니다 — 정지 밝기가 아니라 **시간에 따른 변조**가 정보이기 때문입니다.

> 🌀 **비유.** 선풍기 날개에 손전등을 비추면, 날개가 정면을 보이는 순간마다 규칙적으로 반짝입니다. 그런데 날개가 **선풍기 몸통 뒤로** 넘어가는 동안은 안 보이죠(가림). 이 '보였다 안 보였다'가 규칙적 반짝임을 만듭니다. 몸통 뒤 날개까지 억지로 세면(가림 무시) 밋밋한 몸통 밝기만 커져서 정작 반짝임이 안 보입니다.

> ⚠️ 호버 rpm 은 추력 균형에서 유도한 **가정값**입니다. flash·f_tip 은 rpm 에 비례하므로, 실제 비행 회전수가 다르면 지문 주파수도 그만큼 이동합니다. 지문의 **구조**(깜빡임·±f_tip 경계·기체별 순서)는 믿을 만하나 절대 주파수는 rpm 가정에 달려 있습니다.

In [ ]:
# §1 재현 — 한 드론의 밝기를 한 대역에서 직접 재본다 (SBR)
import numpy as np
from rcs_po import drone_rcs_pattern_bw, dbsm      # 기본 엔진은 'sbr'

az = np.arange(0, 361, 2.0)
sig, n_rays = drone_rcs_pattern_bw('s1000plus', 5.21e9, 80e6, az, el_deg=15.0, n_f=5)
print(f'방위당 광선 {n_rays:,}발  (격자 lambda/16)')
print(f'S1000+ @ 5.2 GHz  방위평균 {dbsm(np.mean(sig)):+.2f} dBsm  '
      f'(로브 최대 {dbsm(np.max(sig)):+.2f} dBsm)')

In [ ]:
# §4 재현 — 호버 rpm 유도 + 블레이드 지문의 두 눈금
#   flash = blades * rpm/60,   f_tip = 2*v_tip/lambda * cos(el)
import numpy as np

specs = dict(mini5pro=(0.2499,0.1524,4), mavic4pro=(1.063,0.267,4),
             matrice4e=(1.219,0.274,4), s1000plus=(9.5,0.381,8),
             phantom4=(1.38,0.240,4))       # (질량 kg, 프로펠러 지름 m, 로터 수)
rho, CT, blades = 1.225, 0.11, 2
lam, el = 3e8/3.5e9, np.deg2rad(15.0)
for d,(m,D,nr) in specs.items():
    T = m*9.81/nr                                  # 로터당 추력 = 무게/로터수
    n = np.sqrt(T/(CT*rho*D**4))                   # T = CT rho n^2 D^4  ->  n [rev/s]
    rpm = n*60
    flash = blades*rpm/60
    v_tip = (2*np.pi*n)*(D/2)
    f_tip = 2*v_tip/lam*np.cos(el)
    print(f'{d:10s} rpm~{rpm:5.0f}  flash {flash:5.1f} Hz  f_tip +-{f_tip/1e3:4.2f} kHz')
# ↑ 여기 나온 rpm 이 report1.json 의 hover_rpm 과 같은 물리에서 나온다

---
## 📌 정리 — 이 리포트가 답한 것

### ❓ "드론 5종은 얼마나 밝고, 프로펠러는 어떤 지문을 남기나?"

**밝기.** 크기가 밝기를 정합니다 — 가장 큰 기체가 가장 작은 기체보다 **10.2 dB** 밝고, 대역(주파수)은 같은 드론을 **1.6 dB** 밖에 못 움직입니다(광학영역). 그리고 그 밝기는 플라스틱 껍데기가 아니라 **속 금속**(모터·배터리·PCB)에서 나옵니다 — 껍데기는 반투명 스크린일 뿐입니다.

**지문.** 프로펠러가 돌면 **120~183 Hz** 의 규칙적 깜빡임과 **±1.0~1.6 kHz** 의 날개끝 도플러가 생깁니다. 이 지문을 보려면 **가림**이 필수입니다 — 몸통 뒤 숨은 날개를 세지 않아야 정지 몸통 신호가 부풀지 않고(순수 PO 는 **9~23 dB** 부풀립니다), 깜빡임이 그 위로 드러납니다.

### 🔬 문헌 실측과 대조 — 우리 절대값이 맞나

우리가 쓰는 신형(Mavic 4 Pro·Matrice 4E)의 실측 RCS 는 아직 논문에 없습니다(2024~25 출시). 대신 **밴드가 겹치는 근접 기종 실측**과 대조하면 우리 값이 타당합니다:

| 문헌 (실측) | 밴드 | 측정 RCS | 우리와의 관계 |
|---|---|---|---|
| **Li & Ling 2017** (IEEE AWPL, ~99인용) | **3–6 GHz** ★밴드일치 | Phantom 2 **−27.5**, 3DR Solo −24.2, Inspire 1 −13.7 dBsm (모두 **peak/특정자세**, 자세 스프레드 ~14 dB) | 우리 mavic4pro **방위평균 −19.7 dBsm → 이 범위 안** (봉우리 자세는 −12.4 dBsm 까지 밝아져 상한 −13.7 을 ~1.3 dB 넘음 — peak↔peak 비교에선 우리가 살짝 밝은 쪽) |
| Ezuma 2019 (compact-range) | 15 / 25 GHz | Phantom 4 Pro −15.0 / −12.4 dBsm | 15→25 GHz 에서 +2.6 dB — RCS 의 주파수 단조증가 방향이 우리 밴드 추세와 일치(절대값 외삽 비교는 밴드갭이 커서 참고 수준) |
| Semkin 2020 (IEEE Access) | 26–40 GHz | Mavic Pro −16.8, Phantom 4 Pro −16.4, **Matrice 100(카본) −10.5** dBsm | **카본이 플라스틱보다 ~7 dB 밝음** — 우리 재질 분해와 방향 일치 |
| Quevedo 2019 (IET RSN) | X-band 8.75 GHz | Phantom 4 −20~−4.6 dBsm(프롭 회전 의존) | 프로펠러가 RCS 를 크게 흔듦 — 우리 마이크로도플러 서사 |

<sub>정리: (1) **같은 밴드(3–6 GHz)** 실측(Phantom 2 −27.5 ~ Inspire 1 −13.7 dBsm)이 우리 **방위평균** −19.7 dBsm 를 감싼다. 단, 봉우리 자세(−12.4 dBsm @3.5 GHz)는 문헌 상한(−13.7)보다 ~1.3 dB 밝다 — 문헌 값도 peak 이므로 과대평가 가능성을 함께 적어 둔다. (2) 고주파 실측들은 모두 더 밝고(RCS 는 주파수↑에 단조↑), 3.5 GHz 로 낮추면 우리 값에 수렴한다. (3) **주의**: 우리 소형드론 값(−17~−26)은 문헌이 링크버짓에 흔히 쓰는 가정치(−10~−13 dBsm)보다 **어두워, 탐지 SNR 을 오히려 보수적으로** 잡는다. 서지: `/data/public/jeong_drone_refs/`.</sub>

### ⚠️ 이 리포트가 **보장하지 않는** 것
- **절대 RCS 값** — 해석해(구·평판)로만 검증(→report07), **이 대역(3.5 GHz)의 드론 실측 앵커는 문헌에도 거의 없다**(위 대조는 근접 기종·다른 밴드). **상대 순서**와 **대역 추세**를 주장하고, 문헌 실측 범위와 **정합**함을 보인다.
- **플라스틱 셸의 정확한 기여** — 반투명이라 '통드론'과 '셸 제거' 사이의 불확실 구간입니다.
- **방위 패턴의 널 깊이** · **절대 회전수** — 인용 금지(§3·§4).

> 🚫 **오해 금지.** "레이트레이싱은 RCS 를 못 낸다" 는 거짓입니다 — 이 리포트의 밝기 숫자 자체가 광선추적(SBR)으로 나왔습니다. 참인 명제는 **"Sionna 기본 solver 에는 산란(PO) 적분이 없어 표적 밝기를 못 낸다"** 뿐이고, 그래서 SBR 을 얹었습니다(→report07).

---
**다음 리포트**: [report09](report09.ipynb) — 이제 **탐지**로 넘어갑니다. 그 전에 챔버 **바닥이 놓는 함정**(표적을 경유해 되돌아오는 유령 신호)을 먼저 봅니다.